<a href="https://colab.research.google.com/github/Sravani-939/genai-training-tasks/blob/main/Sentence_transformer_chunking.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install sentence-transformers faiss-cpu chromadb

In [ ]:
from sentence_transformers import SentenceTransformer
import faiss

# 1. Sample dataset
documents = [
    """Artificial intelligence is a branch of computer science that aims to build
    systems capable of performing tasks that normally require human intelligence.
    These tasks include reasoning, learning, problem solving, and decision-making.
    AI is widely used in healthcare, finance, education, and automation.""",

    """Machine learning is a subset of artificial intelligence. It allows systems
    to learn patterns from data instead of being explicitly programmed for every task.
    Supervised learning, unsupervised learning, and reinforcement learning are common
    machine learning approaches used in modern applications.""",

    """Natural language processing enables computers to understand, interpret, and
    generate human language. It is used in chatbots, sentiment analysis, translation,
    summarization, and question answering systems. NLP combines linguistics with
    machine learning and deep learning techniques.""",

    """Vector databases are used to store embeddings and perform similarity search.
    Tools such as FAISS, Chroma, Milvus, and Weaviate are commonly used in retrieval
    systems. They help applications quickly find text chunks that are semantically
    similar to a user's query.""",

    """Chunking is the process of splitting a large document into smaller pieces.
    Common chunking strategies include fixed-size chunking, sliding window chunking,
    and recursive chunking. Good chunking improves retrieval quality in RAG systems
    because it keeps related information together while staying small enough for search."""
]

def chunk_text(text, chunk_size=80, overlap=20):
    words = text.split()
    chunks = []

    start = 0
    while start < len(words):
        end = start + chunk_size
        chunk = " ".join(words[start:end])
        chunks.append(chunk)

        if end >= len(words):
            break

        start += chunk_size - overlap

    return chunks
all_chunks = []
for doc in documents:
    chunks = chunk_text(doc, chunk_size=80, overlap=20)
    all_chunks.extend(chunks)

print("Total chunks created:", len(all_chunks))
model = SentenceTransformer("all-MiniLM-L6-v2")
embeddings = model.encode(all_chunks)

#faiss search
dimension = embeddings.shape[1]
index = faiss.IndexFlatL2(dimension)
index.add(embeddings)

# output top 3
def retrieve(query, top_k=3):
    query_embedding = model.encode([query])
    distances, indices = index.search(query_embedding, top_k)

    results = []
    for idx in indices[0]:
        results.append(all_chunks[idx])

    return results

query = "Why Faiss is used?"
results = retrieve(query, top_k=3)

print("\nQuery:", query)
print("\nRetrieved Passages:")
for i, passage in enumerate(results, 1):
    print(f"\n{i}. {passage}")